# Hanium AML Colab Setup

Colab Pro에서 LFW 얼굴 분류 모델을 학습하고 targeted FGSM 공격을 실행합니다.

먼저 Google Drive에 아래 파일을 올려두세요.

```text
MyDrive/hanium-aml/archive.zip
```


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. GitHub 코드 가져오기

`%cd`는 Jupyter/Colab의 디렉터리 이동 명령이라 다음 셀에도 위치가 유지됩니다. 셸 명령은 `!`를 붙이면 됩니다.


In [ ]:
!rm -rf /content/hanium-aml
!git clone https://github.com/no-carve-only-pizza/hanium-aml.git /content/hanium-aml
%cd /content/hanium-aml


## 2. 의존성 설치 및 환경 확인


In [ ]:
!pip install -q adversarial-robustness-toolbox[pytorch] foolbox
!python setup_check.py


## 3. Drive의 LFW zip을 Colab 로컬로 풀기

Drive에서 직접 이미지를 읽으면 느릴 수 있으니, zip은 Drive에 보관하고 실행할 때 `/content` 로컬 디스크에 풉니다.


In [ ]:
!mkdir -p data/raw/lfw_zip data/raw
!unzip -q /content/drive/MyDrive/hanium-aml/archive.zip -d data/raw/lfw_zip
!find data/raw/lfw_zip -maxdepth 4 -type d -name 'lfw-deepfunneled' -print


In [ ]:
!rm -f data/raw/lfw
!ln -s /content/hanium-aml/data/raw/lfw_zip/lfw-deepfunneled/lfw-deepfunneled data/raw/lfw
!find data/raw/lfw -type f -name '*.jpg' | wc -l


마지막 출력이 `13233`이면 LFW 연결 성공입니다.


## 4. LFW-10 얼굴 분류 데이터셋 만들기


In [ ]:
!python src/prepare_lfw_identity_dataset.py


## 5. ResNet-50 얼굴 신원 분류 모델 학습

Colab Pro GPU 기준 첫 실행은 12 epoch를 추천합니다. 더 올리고 싶으면 20 epoch까지 봐도 됩니다.


In [ ]:
!python src/train_face_resnet50.py --epochs 12 --batch-size 64 --num-workers 2


## 6. Targeted FGSM 공격 실행

기본값은 각 이미지의 정답 라벨에서 다음 라벨로 가도록 target을 잡습니다. 예: label 0 -> label 1.


In [ ]:
!python src/targeted_fgsm_face.py --epsilon 0.03 --limit 100


## 7. Epsilon sweep


In [ ]:
!for eps in 0.005 0.010 0.030 0.050; do python src/targeted_fgsm_face.py --epsilon "$eps" --limit 100; done


## 8. 결과 요약


In [ ]:
!python src/summarize_face_attack.py


결과 위치:

```text
checkpoints/face_resnet50_lfw10/best.pt
checkpoints/face_resnet50_lfw10/metrics.json
outputs/attacks/fgsm_face/summary.csv
outputs/attacks/fgsm_face/metadata_targeted_eps*.csv
```
